<a href="https://colab.research.google.com/github/MGentieu/dl_project/blob/main/starters/cv-project-starter/cv-project/notebooks/CV_tiny-imagenet-200.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Connexion à github et setup initial

Pour github : à exécuter dans le terminal de colab

```bash
git clone https://github.com/MGentieu/dl_project.git

### Step 0 — On confirme l'utilisation du GPU



In [1]:
!nvidia-smi || echo "nvidia-smi unavailable (CPU runtime)"

Fri Dec  5 20:54:47 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 1 : Installation et importation des bibliothèques

In [2]:
# Installation des librairies nécessaires
!pip -q install torch torchvision torchmetrics matplotlib tqdm ultralytics scikit-learn pyyaml

import os
import sys
import time
import copy
import zipfile
import urllib.request
import shutil
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import Counter
from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score

# Configuration du Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilisation du device : {device}")

# Seeds pour la reproductibilité
torch.manual_seed(42)
np.random.seed(42)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 75.9 MB/s eta 0:00:00
Utilisation du device : cuda


### Step 2 — Pointer le répertoire du projet


In [3]:
# The user has provided the explicit path to the project root.
PROJECT_ROOT = Path("/content/dl_project/starters/cv-project-starter/cv-project")

print(f"Environment: Colab/Kaggle (remote server), using provided PROJECT_ROOT")

# Validate structure
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(f"Missing src/ directory at {PROJECT_ROOT}")

# Setup Python path
os.chdir(PROJECT_ROOT)
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {Path.cwd()}")


Environment: Colab/Kaggle (remote server), using provided PROJECT_ROOT
Project root: /content/dl_project/starters/cv-project-starter/cv-project
Working directory: /content/dl_project/starters/cv-project-starter/cv-project


## M1: Problem Scoping & Data Validation

### 1.1 Définition du problème
L'objectif est de classifier des images de basse résolution. Contrairement à Caltech-101, ce dataset présente un défi plus grand en raison de sa faible résolution native et du nombre plus élevé de classes.

* **Input :** Images RGB (3 canaux). Nativement en 64x64 pixels, elles seront redimensionnées en **224x224** pour correspondre à l'entrée standard de ResNet.
* **Output :** Vecteur de probabilités de taille 200 (Softmax).
* **Métrique d'évaluation :** L'exactitude (Accuracy) sur le jeu de validation sera notre juge de paix.

### 1.2 Data Card — Tiny-ImageNet-200
* **Source :** Sous-ensemble du dataset ImageNet (Stanford CS231n).
* **Volume :** 100 000 images d'entraînement (500/classe) et 10 000 images de validation (50/classe).
* **Caractéristiques :** 200 classes variées (animaux, objets, véhicules).
* **Biais potentiels :** Comme ImageNet, le dataset contient des biais de sélection (occidentaux) et une sur-représentation de certaines catégories (races de chiens).
* **Licence & Usage :** Dataset académique standard pour l'apprentissage.

Le code suivant télécharge les données et restructure le dossier de validation pour le rendre compatible avec `ImageFolder`.

In [4]:
# Téléchargement et préparation (Script fourni dans README_CV_ADVANCED.md)
url = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"
path = "data/tiny-imagenet-200.zip"
data_dir = "data"

if not os.path.exists(os.path.join(data_dir, "tiny-imagenet-200")):
    print("Téléchargement du dataset...")
    os.makedirs(data_dir, exist_ok=True)
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)

    print("Extraction...")
    with zipfile.ZipFile(path, "r") as z:
        z.extractall(data_dir)

    # Restructuration du dossier de validation
    print("Restructuration du dossier de validation...")
    val_dir = pathlib.Path(data_dir) / "tiny-imagenet-200/val"
    val_img_dir = val_dir / "images"

    # Lecture des annotations pour mapper image -> classe
    val_map = {}
    with open(val_dir / "val_annotations.txt", "r") as f:
        for line in f:
            parts = line.strip().split("\t")
            val_map[parts[0]] = parts[1] # image_file -> class_id

    # Déplacement des images dans des sous-dossiers par classe
    for img, wnid in val_map.items():
        target_dir = val_dir / wnid
        target_dir.mkdir(exist_ok=True)
        src = val_img_dir / img
        dst = target_dir / img
        if src.exists():
            shutil.move(src, dst)

    # Suppression du dossier images vide et du fichier txt inutile après déplacement
    if val_img_dir.exists() and not os.listdir(val_img_dir):
        os.rmdir(val_img_dir)

    print("Dataset prêt !")
else:
    print("Le dataset semble déjà présent.")

Téléchargement du dataset...
Extraction...
Restructuration du dossier de validation...
Dataset prêt !


## M2: Baseline Model Implementation

Pour ce problème, nous choisissons l'architecture **ResNet18**.

**Justification du choix :**
1.  **Complexité adaptée :** ResNet18 est suffisamment profond pour capter les features complexes de 200 classes, mais reste léger à entraîner par rapport à ResNet50/101.
2.  **Transfer Learning :** Nous utilisons les poids pré-entraînés sur ImageNet (`IMAGENET1K_V1`). Comme Tiny-ImageNet est un sous-ensemble d'ImageNet, ces poids contiennent déjà des filtres très pertinents (détection de bords, textures, formes).

**Stratégie de la Baseline :**
* Nous **gelons** (freeze) le "backbone" (le extracteur de caractéristiques) pour ne pas détruire les poids pré-entraînés.
* Nous remplaçons uniquement la dernière couche linéaire (Fully Connected) pour adapter la sortie de 1000 à 200 classes.
* Nous effectuons un **Sanity Check** : faire passer un batch dans le modèle avant l'entraînement pour vérifier les dimensions.

In [5]:
# Configuration inspirée de cv_tinyimagenet.yaml
IMG_SIZE = 224
BATCH_SIZE = 128
NUM_WORKERS = 2

# Transformations
# Train : Augmentation de données (Flip, Rotation, ColorJitter) + Resize
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)), # Upscaling de 64x64 à 224x224
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Val : Juste Resize et Normalize
val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Création des Datasets
train_dir = os.path.join(data_dir, "tiny-imagenet-200/train")
val_dir = os.path.join(data_dir, "tiny-imagenet-200/val")

train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
val_dataset = datasets.ImageFolder(root=val_dir, transform=val_transforms)

# Dataloaders
dataloaders = {
    'train': DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS),
    'val': DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
}

dataset_sizes = {'train': len(train_dataset), 'val': len(val_dataset)}
class_names = train_dataset.classes
num_classes = len(class_names)

print(f"Classes: {num_classes}")
print(f"Images Train: {dataset_sizes['train']}, Images Val: {dataset_sizes['val']}")

Classes: 200
Images Train: 100000, Images Val: 10000


In [6]:
def get_model(num_classes, freeze_backbone=True):
    # Chargement de ResNet18 pré-entraîné
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    # 1. Freeze (Geler) le backbone si demandé
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # 2. Modification de la tête (Fully Connected layer)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)

    return model

baseline_model = get_model(num_classes, freeze_backbone=True)
baseline_model = baseline_model.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 162MB/s]


Le chargement du dataset et la transformation basique s'est bien effectuée. On passe donc à l'étape suivante, le chargement réel du dataset et l'exploration des données

On fait ensuite un test pour un batch pour vérifier le bon fonctionnement.

In [7]:
# Récupération d'un batch
inputs, classes = next(iter(dataloaders['train']))
inputs = inputs.to(device)

# Passage dans le modèle
output = baseline_model(inputs)

print(f"Input shape: {inputs.shape}")   # Devrait être [128, 3, 224, 224]
print(f"Output shape: {output.shape}") # Devrait être [128, 200]
print("Test Batch: SUCCESS")

Input shape: torch.Size([128, 3, 224, 224])
Output shape: torch.Size([128, 200])
Test Batch: SUCCESS


## M3: Optimization & Regularization

Nous implémentons ici une boucle d'entraînement complète et robuste intégrant plusieurs techniques de régularisation pour éviter l'overfitting :

1.  **Early Stopping :** Arrêt automatique de l'entraînement si la `val_loss` ne s'améliore pas après un nombre défini d'époques (`patience`). Cela nous assure de garder le meilleur modèle (checkpoint `best.pt`).
2.  **Scheduler (Cosine Annealing) :** Réduction progressive du taux d'apprentissage (Learning Rate) selon une courbe cosinus. Cela permet de converger rapidement au début puis d'affiner les poids en douceur vers la fin.
3.  **Optimiseur (AdamW) :** Une variante de Adam avec une meilleure gestion du *Weight Decay* (L2 Regularization), essentielle pour la généralisation sur des modèles pré-entraînés.

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer_base = optim.Adam(baseline_model.fc.parameters(), lr=1e-3)

print("Lancement de la baseline (1 époque)...")
# Boucle simplifiée pour M2
baseline_model.train()
running_loss = 0.0
for i, (inputs, labels) in enumerate(dataloaders['train']):
    inputs, labels = inputs.to(device), labels.to(device)

    optimizer_base.zero_grad()
    outputs = baseline_model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer_base.step()

    running_loss += loss.item()
    if i % 100 == 0:
        print(f"Batch {i}, Loss: {loss.item():.4f}")

print("Baseline training finished.")

Lancement de la baseline (1 époque)...
Batch 0, Loss: 5.5142
Batch 100, Loss: 4.1680
Batch 200, Loss: 3.3837
Batch 300, Loss: 3.0985
Batch 400, Loss: 3.0352
Batch 500, Loss: 2.5666
Batch 600, Loss: 2.4513
Batch 700, Loss: 2.0337
Baseline training finished.


In [9]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = np.inf
        self.early_stop = False

    def __call__(self, val_loss, model, path='best_model.pt'):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), path)
            print(f"Validation loss improved to {val_loss:.4f}. Model saved.")
        else:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

def train_model(model, criterion, optimizer, scheduler, num_epochs=25, patience=5):
    since = time.time()
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    early_stopping = EarlyStopping(patience=patience)

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        # Chaque époque a une phase d'entraînement et une phase de validation
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                # Forward
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward + Optimize uniquement si train
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Enregistrement historique
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.cpu().item())
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.cpu().item())

                # Check Early Stopping
                early_stopping(epoch_loss, model)

        if early_stopping.early_stop:
            print("Early stopping triggered")
            break

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')

    # Charger le meilleur modèle
    model.load_state_dict(torch.load('best_model.pt'))
    return model, history

## M4: Ablation Studies & Analysis

Pour valider nos choix techniques, nous menons une étude comparative rigoureuse (Ablation Study). L'objectif est de mesurer l'impact de la stratégie de **Fine-Tuning** (affinement).

Nous comparons trois approches via le script automatisé ci-dessous :

1.  **Baseline (Frozen Only) :** On entraîne uniquement la tête de classification. Le backbone reste gelé. C'est rapide, mais cela limite la capacité du modèle à s'adapter aux spécificités des images 64x64 upscalées.
2.  **Stratégie Complète (Frozen $\to$ Fine-Tune) :**
    * *Phase 1 :* Entraînement de la tête (Backbone gelé).
    * *Phase 2 :* On dégèle tout le réseau et on entraîne avec un Learning Rate très faible (`1e-5`). Cela permet d'ajuster finement les filtres convolutionnels sans "casser" les connaissances acquises.
3.  **Full Training (Direct) :** Tout le réseau est entraîné dès le départ. C'est une stratégie risquée qui peut mener à l'oubli catastrophique des poids pré-entraînés si le Learning Rate est trop haut.

Les résultats seront présentés dans un tableau comparatif final.

In [ ]:
import copy
import pandas as pd

# Fonction de mise à jour de config
def update_recursive(d, u):
    for k, v in u.items():
        if isinstance(v, dict):
            d[k] = update_recursive(d.get(k, {}), v)
        else:
            d[k] = v
    return d

# --- DÉFINITION DES EXPÉRIENCES ---
# C'est ici qu'on définit si on veut les 2 phases ou non
experiments_config = {
    "1. Baseline (Frozen Only)": {
        "updates": {
            "train": {"epochs": 10},
            "model": {"freeze_backbone": True}
        },
        "fine_tune_after": False, # Pas de phase 2
        "description": "Entraînement rapide, uniquement la dernière couche."
    },
    "2. Strategie Complete (Frozen -> Fine-Tune)": {
        "updates": {
            "train": {"epochs": 8}, # Phase 1 un peu plus courte
            "model": {"freeze_backbone": True}
        },
        "fine_tune_after": True, # <--- ACTIVE LA PHASE 2
        "fine_tune_params": {"lr": 1e-5, "epochs": 8},
        "description": "Stratégie optimale : Transfert puis affinage complet."
    },
    "3. Full Training (Direct)": {
        "updates": {
            "train": {"epochs": 15, "lr": 1e-4},
            "model": {"freeze_backbone": False} # Tout est dégelé dès le début
        },
        "fine_tune_after": False,
        "description": "Entraînement de tout le réseau directement (risqué)."
    }
}

# Config par défaut
base_config = {
    'model': {'pretrained': True, 'freeze_backbone': True, 'num_classes': 200},
    'train': {'lr': 5e-4, 'epochs': 10, 'weight_decay': 1e-2},
    'device': device
}

results_table = []

print(f"{'='*60}\nLANCEMENT DES ÉTUDES D'ABLATION (M4 - AVEC 2 PHASES)\n{'='*60}")

for exp_name, exp_data in experiments_config.items():
    print(f"\n>>> Expérience : {exp_name}")
    print(f"Description : {exp_data['description']}")

    # 1. Configuration
    current_config = copy.deepcopy(base_config)
    update_recursive(current_config, exp_data['updates'])

    # 2. Setup Modèle (Phase 1)
    model_abl = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if current_config['model']['pretrained'] else None)

    # Freeze backbone selon la config
    if current_config['model']['pretrained'] and current_config['model']['freeze_backbone']:
        for param in model_abl.parameters():
            param.requires_grad = False

    num_ftrs = model_abl.fc.in_features
    model_abl.fc = nn.Linear(num_ftrs, current_config['model']['num_classes'])
    model_abl = model_abl.to(device)

    # 3. Entraînement Phase 1
    print("--- Phase 1 ---")
    params_to_update = [p for p in model_abl.parameters() if p.requires_grad]
    optimizer = optim.AdamW(params_to_update, lr=current_config['train']['lr'], weight_decay=current_config['train']['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=current_config['train']['epochs'])

    model_abl, hist1 = train_model(model_abl, criterion, optimizer, scheduler,
                                   num_epochs=current_config['train']['epochs'], patience=3)

    best_acc = max(hist1['val_acc'])
    final_loss = hist1['val_loss'][-1]

    # 4. Entraînement Phase 2 (Optionnel : Fine-Tuning)
    if exp_data.get("fine_tune_after", False):
        print("\n--- Phase 2 : Décongélation & Fine-Tuning ---")

        # On dégèle tout
        for param in model_abl.parameters():
            param.requires_grad = True

        ft_params = exp_data.get("fine_tune_params", {"lr": 1e-5, "epochs": 5})

        # Nouvel optimiseur avec LR plus faible
        optimizer_ft = optim.AdamW(model_abl.parameters(), lr=ft_params["lr"], weight_decay=1e-2)
        scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=ft_params["epochs"])

        model_abl, hist2 = train_model(model_abl, criterion, optimizer_ft, scheduler_ft,
                                       num_epochs=ft_params["epochs"], patience=3)

        # Mise à jour des metrics avec la phase 2
        best_acc = max(max(hist1['val_acc']), max(hist2['val_acc']))
        final_loss = hist2['val_loss'][-1]

    # Enregistrement
    results_table.append({
        "Experiment": exp_name,
        "Phase 2 (FT)": "Oui" if exp_data.get("fine_tune_after") else "Non",
        "Best Val Acc": best_acc,
        "Final Val Loss": final_loss
    })

# Affichage
df_results = pd.DataFrame(results_table)
print("\n=== RÉSULTATS COMPARATIFS ===")
display(df_results)

LANCEMENT DES ÉTUDES D'ABLATION (M4 - AVEC 2 PHASES)

>>> Expérience : 1. Baseline (Frozen Only)
Description : Entraînement rapide, uniquement la dernière couche.
--- Phase 1 ---
Epoch 1/10
----------
train Loss: 3.5933 Acc: 0.2902
val Loss: 2.4073 Acc: 0.4725
Validation loss improved to 2.4073. Model saved.
Epoch 2/10
----------
train Loss: 2.5274 Acc: 0.4398
val Loss: 2.0469 Acc: 0.5145
Validation loss improved to 2.0469. Model saved.
Epoch 3/10
----------
train Loss: 2.2936 Acc: 0.4693
val Loss: 1.9110 Acc: 0.5387
Validation loss improved to 1.9110. Model saved.
Epoch 4/10
----------
train Loss: 2.1935 Acc: 0.4858
val Loss: 1.8394 Acc: 0.5503
Validation loss improved to 1.8394. Model saved.
Epoch 5/10
----------
train Loss: 2.1338 Acc: 0.4949
val Loss: 1.8043 Acc: 0.5618
Validation loss improved to 1.8043. Model saved.
Epoch 6/10
----------
train Loss: 2.0893 Acc: 0.5053
val Loss: 1.7797 Acc: 0.5634
Validation loss improved to 1.7797. Model saved.
Epoch 7/10
----------
train Loss: 2

## M5: Reporting & Final Delivery

### 5.1 Analyse des performances
Nous évaluons ici le meilleur modèle obtenu lors de l'étude d'ablation (la stratégie "Frozen -> Fine-Tune").

Nous utilisons les métriques suivantes :
* **Classification Report :** Précision, Rappel et F1-Score (macro-average) pour avoir une vue d'ensemble.
* **Matrice de Confusion :** Pour visualiser les erreurs. Étant donné les 200 classes, nous affichons un sous-ensemble (les 20 premières classes) pour que le graphique reste lisible. Une diagonale forte indique de bonnes prédictions.

### 5.2 Analyse des échecs
Les erreurs sur Tiny-ImageNet proviennent souvent de deux facteurs :
1.  **Résolution :** L'upscaling de 64x64 à 224x224 crée du flou, rendant difficile la distinction de classes visuellement proches (ex: différentes races de chiens ou d'oiseaux).
2.  **Similarité sémantique :** Le modèle peut confondre des objets qui partagent des textures similaires (ex: confusion entre "Table" et "Bureau").

L'analyse ci-dessous permet de confirmer ces hypothèses.

In [ ]:
results_df = pd.DataFrame({
    'Expérience': ['Frozen Head Only', 'Fine-Tuning'],
    'Best Val Acc': [max(hist1['val_acc']), max(hist2['val_acc'])],
    'Final Val Loss': [hist1['val_loss'][-1], hist2['val_loss'][-1]]
})
display(results_df)

In [ ]:
def evaluate_model_detailed(model, dataloader):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    return np.array(y_true), np.array(y_pred)

# Evaluation
y_true, y_pred = evaluate_model_detailed(model_final, dataloaders['val'])

# Rapport de classification
print(classification_report(y_true, y_pred, digits=4))

# Matrice de confusion (Sur un sous-ensemble de classes pour lisibilité)
# On affiche les 20 premières classes
plt.figure(figsize=(12, 10))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm[:20, :20], annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix (First 20 classes)")
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()